# Classification after-class exercise — Design a maintenance alarm
### Individual exercise 

You support a reliability team monitoring industrial rotating equipment. Failures are rare, a missed failure costs approximately **£20,000**, and an unnecessary inspection costs **£1,000**. Your task is to design and justify an alarm policy—not merely maximise accuracy.

**Learning outcomes**

1. Diagnose the accuracy trap in imbalanced classification.
2. Interpret precision, recall, F1, ROC-AUC, PR-AUC, and a confusion matrix.
3. Compare model families on validation data and use physics-based features.
4. Select an alarm threshold from costs, then evaluate once on untouched test data.




## 0. Setup 

Run the next two cells without editing them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay,
)

from exercise_data import load_ai4i

def metric_row(y_true, prediction, probability):
    return {
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc": average_precision_score(y_true, probability),
    }

plt.rcParams["figure.figsize"] = (8, 4)
print("Setup complete")

In [ ]:
# Keep exercise_data.py in the same folder as this notebook.
# Keep this set to True during the exercise.
# The notebook will download the published dataset.
PREFER_REAL_DATA = True
pm = load_ai4i(prefer_real=PREFER_REAL_DATA)
print("Source:", pm.attrs["source"])
pm.head()

## 1. The accuracy trap 

The AI4I predictive-maintenance dataset contains temperature, speed, torque, tool wear, machine type, and five failure-mode flags. First determine how rare failures are and test the apparently impressive policy “always predict healthy.”

In [ ]:
FAILURE_MODES = ["TWF", "HDF", "PWF", "OSF", "RNF"]

# TODO 1: calculate the failure rate and number of each failure mode.
failure_rate = ...
mode_counts = ...

# TODO 2: evaluate a rule that always predicts "healthy" (class 0).
always_healthy = ...
baseline_accuracy = ...
baseline_recall = ...

print(f"Failure rate: {failure_rate:.3%}")
print("Failures by mode:")
display(mode_counts.to_frame("count"))
print(f"Always-healthy accuracy: {baseline_accuracy:.3f}")
print(f"Always-healthy recall:   {baseline_recall:.3f}")

assert 0 < failure_rate < 0.15
assert baseline_recall == 0



### Written answer 1

Why is the always-healthy accuracy misleading? What two pieces of information should accompany any accuracy claim on an imbalanced problem?

> **Your answer:**

## 2. Reserve validation and final-test data 

Use training data to fit, validation data to compare and tune, and final-test data once for the final claim.

In [ ]:
RAW_FEATURES = ["air_temp", "proc_temp", "speed", "torque", "tool_wear"]
X = pm[RAW_FEATURES].copy()
X = pd.concat(
    [X, pd.get_dummies(pm["type"], prefix="type", drop_first=True, dtype=int)],
    axis=1,
)
y = pm["fail"].copy()

# TODO 3: make stratified 60% training, 20% validation and 20% final-test sets.
X_train, X_holdout, y_train, y_holdout = ...
X_val, X_test, y_val, y_test = ...

print("rows:", len(X_train), "train |", len(X_val), "validation |", len(X_test), "test")
print("failure rates:", y_train.mean().round(3), y_val.mean().round(3), y_test.mean().round(3))

assert len(X_train) + len(X_val) + len(X_test) == len(pm)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

## 3. First classifier and confusion matrix 

In [ ]:
# TODO 4: build a StandardScaler + LogisticRegression(max_iter=2000) pipeline.
plain_logit = ...
plain_logit.fit(...)
val_prediction = ...
val_probability = ...
plain_metrics = metric_row(y_val, val_prediction, val_probability)

display(pd.Series(plain_metrics, name="plain logistic").round(3))
ConfusionMatrixDisplay.from_predictions(y_val, val_prediction, display_labels=["healthy", "failure"])
plt.title("Plain logistic regression — validation set")
plt.show()

assert set(plain_metrics) == {"accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"}

## 4. Compare model families 

Compare a reweighted linear boundary with two threshold-based model families. Use validation data only.

In [ ]:
# TODO 5: complete these three model definitions.
models = {
    "balanced logistic": ...,
    "decision tree": ...,
    "random forest": ...,
}

validation_rows = []
for name, model in models.items():
    # TODO 6: fit on training data and evaluate on validation data.
    model.fit(...)
    prediction = ...
    probability = ...
    validation_rows.append({"model": name, **metric_row(y_val, prediction, probability)})

model_results = pd.DataFrame(validation_rows).set_index("model")
display(model_results.round(3))

assert len(model_results) == 3
assert np.isfinite(model_results.to_numpy()).all()

<details><summary>Hints for TODOs 5–6</summary>

- Balanced logistic: a scaled pipeline with `class_weight="balanced"`.
- Tree: `DecisionTreeClassifier(max_depth=6, random_state=0)`.
- Forest: `RandomForestClassifier(n_estimators=200, random_state=0)`.

</details>

### Written answer 3

Which model has the strongest ranking performance? Which produces the most false-alarm pressure? Explain why accuracy, ROC-AUC, and PR-AUC answer different questions.

> **Your answer:**

## 5. Put process physics into the features 

Several failure mechanisms depend on combinations rather than raw measurements. Mechanical power is (P=\tau\omega); heat removal depends on the process-to-air temperature difference; overstrain depends on load accumulated with wear.

In [ ]:
def add_physics_features(frame):
    result = frame.copy()
    # TODO 7: add mechanical power [kW], process-to-air temperature difference,
    # and wear × torque (scaled by 1000 only to keep the numbers compact).
    result["power_kw"] = ...
    result["temperature_gap"] = ...
    result["wear_x_torque"] = ...
    return result

X_physics = add_physics_features(X)
X_physics_train = X_physics.loc[X_train.index]
X_physics_val = X_physics.loc[X_val.index]

physics_logit = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")
)
physics_logit.fit(X_physics_train, y_train)
physics_prediction = physics_logit.predict(X_physics_val)
physics_probability = physics_logit.predict_proba(X_physics_val)[:, 1]
physics_metrics = metric_row(y_val, physics_prediction, physics_probability)

comparison = pd.DataFrame(
    {
        "balanced logistic — raw": model_results.loc["balanced logistic"],
        "balanced logistic — physics": pd.Series(physics_metrics),
    }
).T
display(comparison.round(3))

assert {"power_kw", "temperature_gap", "wear_x_torque"}.issubset(X_physics.columns)

<details><summary>Hint for TODO 7</summary>

Use `torque * speed * 2*np.pi/60/1000`, `proc_temp - air_temp`, and `tool_wear * torque / 1000`.

</details>

### Written answer 4

How did the physical features change logistic regression's validation performance? Why can feature construction matter more than changing the algorithm?

> **Your answer:**

## 6. Select an alarm threshold 

The forest is used as the candidate alarm model. Select its threshold using validation costs—not final-test costs.

In [ ]:
C_MISS = 20_000
C_FALSE_ALARM = 1_000
alarm_model = models["random forest"]
validation_probability = alarm_model.predict_proba(X_val)[:, 1]

thresholds = [0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70]
cost_rows = []

for threshold in thresholds:
    # TODO 8: convert probabilities to alarms and calculate FN, FP, recall,
    # precision, and total cost on the validation set.
    alarm = ...
    tn, fp, fn, tp = ...
    cost = ...
    cost_rows.append(
        {
            "threshold": threshold,
            "missed_failures": fn,
            "false_alarms": fp,
            "recall": ...,
            "precision": ...,
            "cost_GBP": cost,
        }
    )

cost_table = pd.DataFrame(cost_rows).set_index("threshold")
best_threshold = float(cost_table["cost_GBP"].idxmin())
display(cost_table.round(3))
print("Selected validation threshold:", best_threshold)

assert best_threshold in thresholds

<details><summary>Hint for TODO 8</summary>

Use `(validation_probability >= threshold).astype(int)`, then `confusion_matrix(y_val, alarm, labels=[0,1]).ravel()`. Cost is `fn*C_MISS + fp*C_FALSE_ALARM`.

</details>

Why is the selected threshold below 0.5? What practical problem appears when the threshold is pushed too low?

> **Your answer:**

## 7. Final test and recommendation 

The threshold is fixed. Open the final test set once and report the operational result.

In [ ]:
# This is the first and only model-evaluation section that uses the final test set.
test_probability = alarm_model.predict_proba(X_test)[:, 1]

# TODO 9: apply the validation-selected threshold to the test probabilities.
test_alarm = ...
tn, fp, fn, tp = ...
test_cost = ...
test_metrics = metric_row(y_test, test_alarm, test_probability)

default_alarm = (test_probability >= 0.50).astype(int)
_, default_fp, default_fn, _ = confusion_matrix(y_test, default_alarm, labels=[0, 1]).ravel()
default_cost = default_fn * C_MISS + default_fp * C_FALSE_ALARM

display(pd.Series(test_metrics, name="final test").round(3))
print(f"Selected threshold: {best_threshold:.2f}")
print(f"Missed failures: {fn} | False alarms: {fp}")
print(f"Test cost: £{test_cost:,.0f}")
print(f"Default-0.50 test cost: £{default_cost:,.0f}")

ConfusionMatrixDisplay.from_predictions(
    y_test, test_alarm, display_labels=["healthy", "failure"]
)
plt.title("Final test set at the validation-selected threshold")
plt.show()

assert test_alarm.shape == y_test.shape

<details><summary>Hint for TODO 9</summary>

Apply `best_threshold` exactly as in the validation loop. Extract the confusion-matrix counts and calculate the same cost expression.

</details>

### Final recommendation — maximum 150 words

Write to the maintenance manager. State:

- the model and threshold;
- test recall, precision, missed failures, and false alarms;
- test cost versus the default 0.5 threshold;
- why accuracy alone is unsuitable;
- one validation requirement before using the policy on a real fleet.

> **Your recommendation:**



## Optional classification extensions

All comparisons below use `X_train` for fitting and `X_val` for evaluation.

### A. Gradient-boosted trees 

Boosting builds trees sequentially so each learner focuses on mistakes left by the current ensemble. This is the central idea behind XGBoost, LightGBM and CatBoost.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

boosted_classifier = GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=3, random_state=0
)

# OPTIONAL TASK: fit on training data and evaluate on validation data using metric_row.
boosted_classifier.fit(...)
boosted_prediction = ...
boosted_probability = ...
boosted_metrics = ...
display(pd.Series(boosted_metrics, name="gradient boosting").round(3))

# OPTIONAL QUESTION: Compare boosting with the forest. How do sequential boosting
# and independent-tree averaging differ conceptually?

### B. kNN, SVM and naïve Bayes 

- **kNN** predicts from nearby observations, so feature scaling is essential.
- **SVM** searches for a wide separating margin; an RBF kernel creates nonlinear boundaries.
- **Gaussian naïve Bayes** models each feature distribution within each class and assumes conditional independence.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB

optional_classifiers = {
    "kNN (k=11)": make_pipeline(StandardScaler(), KNeighborsClassifier(11)),
    "RBF SVM": make_pipeline(
        StandardScaler(),
        CalibratedClassifierCV(
            SVC(kernel="rbf", class_weight="balanced"),
            method="sigmoid", cv=3,
        ),
    ),
    "Gaussian naïve Bayes": GaussianNB(),
}

optional_rows = []
for name, model in optional_classifiers.items():
    # OPTIONAL TASK: fit on training data and evaluate on validation data.
    model.fit(...)
    prediction = ...
    probability = ...
    optional_rows.append({"model": name, **metric_row(y_val, prediction, probability)})

optional_results = pd.DataFrame(optional_rows).set_index("model")
display(optional_results.round(3))

# OPTIONAL QUESTION: Which model is most affected by scaling? Which modelling
# assumption made by naïve Bayes is least plausible for these sensor variables?

# Supervised-learning algorithm map

This is a navigation guide, not a checklist of algorithms that must all be used. A strong workflow starts with a baseline, chooses a model family suited to the data, validates honestly, and only adds complexity when it improves the decision being made.

## Core families

### Concepts and coverage

| Family | Famous algorithms | Main idea | Where used here |
|---|---|---|---|
| Baselines | Mean predictor, majority class | Establish performance achieved without learning useful structure | Both required exercises |
| Linear regression | Ordinary least squares | Fit a weighted sum to a continuous target | Required regression |
| Regularised linear models | Ridge, Lasso, Elastic Net | Penalise large coefficients; Lasso may set some to zero | Optional regression |
| Linear classification | Logistic regression | Model class probability through a linear decision boundary | Required classification |
| Nearest neighbours | kNN | Predict from nearby training observations | Optional classification |
| Single trees | Decision tree | Recursively split variables into rule-like regions | Both required exercises |
| Bagged trees | Random forest | Average many decorrelated trees | Required classification |
| Boosted trees | Gradient Boosting, XGBoost, LightGBM, CatBoost | Build trees sequentially to correct earlier errors | Optional in both exercises |
| Support vector machines | Linear SVM, kernel SVM | Maximise the margin; kernels create nonlinear boundaries | Optional classification |
| Probabilistic classifiers | Gaussian, Multinomial and Bernoulli naïve Bayes | Combine feature likelihoods under a conditional-independence assumption | Optional classification |
| Neural networks | Multi-layer perceptron, deep networks | Learn successive nonlinear feature transformations | Required regression |

### Practical characteristics

| Family | Scaling? | Particularly useful when | Main caution |
|---|---|---|---|
| Baselines | No | Every problem | A sophisticated model is not valuable unless it beats the baseline |
| Linear regression | Usually not essential | Relationships are approximately linear and interpretation matters | Misses nonlinearities; coefficients can be misleading under collinearity |
| Regularised linear models | Yes | Features are numerous, correlated, or noisy | Regularisation strength must be selected by validation |
| Linear classification | Usually yes | An interpretable classifier or strong baseline is needed | Raw linear boundaries cannot represent interactions automatically |
| Nearest neighbours | Yes | Local geometry is meaningful and the dataset is not too large | Sensitive to scaling, irrelevant variables, and high dimension |
| Single trees | No | Nonlinear thresholds and interactions matter | Deep trees overfit easily |
| Bagged trees | No | Strong general-purpose tabular performance is needed | Less interpretable; probabilities may need calibration |
| Boosted trees | No | High predictive accuracy on structured or tabular data matters | More tuning; careless validation can overfit |
| Support vector machines | Yes | A medium-sized dataset has clear class separation | Kernel choice and hyperparameters matter; probabilities are indirect |
| Probabilistic classifiers | Usually no | A fast baseline is needed and class-conditional models are plausible | Conditional independence can be unrealistic |
| Neural networks | Yes | Complex nonlinear structure and sufficient data are available | Optimisation, scaling, and architecture choices matter |

## A practical selection sequence

1. Establish a task-appropriate baseline.
2. Fit an interpretable linear or logistic model.
3. Inspect validation errors and residuals for missing nonlinear structure.
4. Try a tree-based model for thresholds and interactions.
5. For tabular data, compare a random forest or gradient-boosted trees.
6. Use kNN or an SVM when distances or margins are meaningful and features can be scaled.
7. Use a neural network when nonlinear capacity is justified by the data and validation evidence.
8. Select the final operating threshold or regression model before opening the test set.

## Important distinctions

- **Regression versus classification** describes the target, not the model's sophistication. Trees, boosting, kNN, SVMs and neural networks have variants for both.
- **Bagging** trains models largely independently and averages them; random forests are the standard example.
- **Boosting** trains models sequentially so later learners focus on earlier errors.
- **Probability estimation and decisions are different.** A classifier ranks or estimates risk; an operational threshold converts that output into an action.
- **XGBoost, LightGBM and CatBoost** are widely used boosting libraries. The optional notebooks use scikit-learn's gradient boosting so no additional dependency is required; the underlying sequential-boosting idea is the transferable lesson.

## What remains outside these exercises

The package does not attempt a full treatment of multiclass classification, probability calibration, feature selection, uncertainty intervals, explainability methods, data drift, time-series validation, or deep-learning architectures. These are natural follow-on topics after students understand the workflow in the two required exercises.
